In [1]:
# 1. Remove existing repo and any cached packages
!rm -rf /content/water-quality-qubo

# 2. Clone fresh
!git clone https://github.com/carlers/water-quality-qubo.git /content/water-quality-qubo

# 3. Change working directory to repo root
import os
os.chdir('/content/water-quality-qubo')
print("Current working directory:", os.getcwd())

# 4. Install packages with verbose output
%pip install -r requirements.txt

# 5. Add repo to Python path
import sys
sys.path.append('/content/water-quality-qubo')

Cloning into '/content/water-quality-qubo'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 50 (delta 16), reused 37 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 12.71 KiB | 12.71 MiB/s, done.
Resolving deltas: 100% (16/16), done.
Current working directory: /content/water-quality-qubo


In [2]:
import numpy as np
print("NumPy version:", np.__version__)

import jijmodeling as jm
print("JijModeling version:", jm.__version__)

# Try different import for transpiler (newer versions may have it as a submodule)
try:
    import jijmodeling_transpiler as jmt
    print("Imported jijmodeling_transpiler successfully")
except ImportError:
    try:
        from jijmodeling import transpiler as jmt
        print("Imported transpiler from jijmodeling successfully")
    except ImportError:
        print("Could not import transpiler. Please check installation.")

import openjij as oj
print("OpenJij imported")

import cvxpy as cp
print("CVXPY imported")

print("All imports successful!")

NumPy version: 1.26.4
JijModeling version: 1.14.2
Imported jijmodeling_transpiler successfully
OpenJij imported
CVXPY imported
All imports successful!


In [4]:
# Cell 3: Complete Test (N=10, K=5) with exact QUBO solver
import numpy as np
from config import CONFIG
from data.synthetic_data import (
    generate_hexagonal_lattice, generate_utility_scores,
    build_pairwise_terms, select_random_sites
)
from src.model import build_water_quality_problem, transpile_qubo
from src.solvers import solve_exact_qubo, solve_sa, solve_sqa
from src.utils import compute_sqr

# 1. Parameters
N = 10
K = 5
config = CONFIG
num_trials = 5  # small for testing

# 2. Generate data
sites = generate_hexagonal_lattice(config['d_nn'], config['domain_size'])
sites = select_random_sites(sites, N)
U = generate_utility_scores(sites, config['ahp_weights'])
a_i = -U
b_ij = build_pairwise_terms(sites, config)

# 3. Tune lambda (strong penalty)
max_abs_a = np.max(np.abs(a_i))
max_row_sum = np.max(np.sum(np.abs(b_ij), axis=1))
lambda_penalty = 50.0 * (max_abs_a + max_row_sum)   # 50× bound
print(f"Lambda = {lambda_penalty:.2f}")

# 4. Build QUBO
prob = build_water_quality_problem(N, K, a_i, b_ij, lambda_penalty)
instance_data = {'a': a_i, 'b': b_ij, 'lambda': lambda_penalty, 'K': K}
qubo, consts = transpile_qubo(prob, instance_data)
print(f"QUBO has {len(qubo)} terms")

# 5. Exact solution (dimod)
x_exact, energy_exact = solve_exact_qubo(qubo, N, K)
print(f"Exact: selected {np.sum(x_exact)} stations, energy = {energy_exact:.4f}")
print(f"Exact indices: {np.where(x_exact == 1)[0]}")

# 6. SA (multiple trials)
energies_sa = []
x_best_sa = None
for trial in range(num_trials):
    x_sa, energy_sa = solve_sa(qubo, num_reads=1, sweeps=500)
    energies_sa.append(energy_sa)
    if energy_sa < min(energies_sa):
        x_best_sa = x_sa
best_energy_sa = min(energies_sa)
print(f"SA: selected {np.sum(x_best_sa)} stations, best energy = {best_energy_sa:.4f}")
print(f"SA indices: {np.where(x_best_sa == 1)[0]}")

# 7. SQA (multiple trials)
energies_sqa = []
x_best_sqa = None
for trial in range(num_trials):
    x_sqa, energy_sqa = solve_sqa(qubo, num_reads=1, sweeps=500, trotter=8)
    energies_sqa.append(energy_sqa)
    if energy_sqa < min(energies_sqa):
        x_best_sqa = x_sqa
best_energy_sqa = min(energies_sqa)
print(f"SQA: selected {np.sum(x_best_sqa)} stations, best energy = {best_energy_sqa:.4f}")
print(f"SQA indices: {np.where(x_best_sqa == 1)[0]}")

# 8. Check feasibility
feasible_sa = (np.sum(x_best_sa) == K)
feasible_sqa = (np.sum(x_best_sqa) == K)
print(f"SA feasible: {feasible_sa}, SQA feasible: {feasible_sqa}")

# 9. SQR
if feasible_sa:
    sqr_sa = compute_sqr(best_energy_sa, energy_exact)
    print(f"SA SQR = {sqr_sa:.4f}")
if feasible_sqa:
    sqr_sqa = compute_sqr(best_energy_sqa, energy_exact)
    print(f"SQA SQR = {sqr_sqa:.4f}")

ModuleNotFoundError: No module named 'config'

In [3]:
!nvidia-smi

Thu Jun 25 08:31:38 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----